# Zenith v31 — Neural Cardiac Age Clock Training

This notebook trains the **Neural Age Clock** — a companion to the Horvath epigenetic clock that predicts the biological age of the **cardiac nervous system** (intrinsic cardiac neurons + vagal innervation).

## Scientific Basis

- The heart has ~40,000 intrinsic neurons (Ardell, 2004)
- Vagal tone declines ~1.5%/year after age 40 (Umetani et al., 1998)
- Heart Rate Variability (HRV) is the clinical correlate
- Key markers: CHAT (parasympathetic), TH (sympathetic), NGFR (neuronal health), CHRNA7 (vagal)

## What This Notebook Does

1. Loads the **Human Heart Cell Atlas** (same data as Zenith v26)
2. Trains the scVI foundation model (if not already trained)
3. Extracts neural marker genes from the scVI latent space
4. Fits the Neural Age Clock regression on donor-age-labeled cells
5. Validates against the Horvath clock (dual-age assessment)
6. Saves the trained clock for use in `bridge_server.py`

**Hardware**: Google Colab T4/L4/A100 (same as v26/v28 training)
**Runtime**: ~25 minutes on T4

## 1. Environment Setup

In [ ]:
# Install dependencies (same as v26 training)
!pip install scanpy scvi-tools anndata torch numpy pandas matplotlib -q

import scanpy as sc
import scvi
import anndata
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

scvi.settings.seed = 42
print(f"scvi-tools version: {scvi.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Human Heart Cell Atlas

Same dataset as Zenith v26 (Litviňuková et al., Nature 2020).
486,134 adult cardiac cells with donor age metadata.

In [ ]:
# Download the Human Heart Cell Atlas (if not present)
# This is the same h5ad used in Zenith_v26_486k_Training.ipynb
DATA_PATH = "heart_cell_atlas.h5ad"

if not os.path.exists(DATA_PATH):
    print("Downloading Human Heart Cell Atlas...")
    # Replace with your actual data source URL
    # !wget -O {DATA_PATH} "https://..."
    print("NOTE: Place your h5ad file here or update the URL")
else:
    print(f"Loading {DATA_PATH}...")

adata = sc.read_h5ad(DATA_PATH)
print(f"Loaded: {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"Obs columns: {list(adata.obs.columns)[:10]}")

In [ ]:
# Check for donor age metadata (critical for neural age training)
age_col = None
for col in ['age', 'donor_age', 'Age', 'patient_age', 'age_group']:
    if col in adata.obs.columns:
        age_col = col
        break

if age_col:
    ages = adata.obs[age_col]
    print(f"Age column: '{age_col}'")
    print(f"  Range: {ages.min()} - {ages.max()}")
    print(f"  Mean: {ages.mean():.1f}")
    print(f"  Unique donors: {adata.obs.get('donor_id', adata.obs.get('sample', 'unknown')).nunique()}")
else:
    print("WARNING: No age column found. Neural age clock needs donor age labels.")
    print("Available columns:", list(adata.obs.columns))

## 3. Preprocessing & HVG Selection (same as v26)

In [ ]:
# Keep raw counts in a layer
adata.layers['counts'] = adata.X.copy()

# Highly Variable Genes (top 4000, seurat_v3)
sc.pp.highly_variable_genes(adata, n_top_genes=4000, flavor='seurat_v3', batch_key='dataset_id' if 'dataset_id' in adata.obs.columns else None)
print(f"HVG selected: {adata.var['highly_variable'].sum()} genes")

# Verify neural markers are in HVG
NEURAL_MARKERS = ['CHAT', 'SLC18A3', 'CHRNA7', 'CHRM2', 'TH', 'DBH', 'NGFR', 'RET', 'NTRK1', 'PHOX2B', 'ISL1', 'GJA1', 'GJA5']
found = [g for g in NEURAL_MARKERS if g in adata.var_names and adata.var.loc[g, 'highly_variable']]
missing = [g for g in NEURAL_MARKERS if g not in adata.var_names or not adata.var.loc[g, 'highly_variable']]
print(f"Neural markers in HVG: {len(found)}/{len(NEURAL_MARKERS)}")
if missing:
    print(f"  Missing/not-HVG: {missing}")
    # Force-include missing markers
    for g in missing:
        if g in adata.var_names:
            adata.var.loc[g, 'highly_variable'] = True
            print(f"  Force-included: {g}")

## 4. Train scVI Foundation Model (or load existing v26)

If you already have `models/scvi_model_486k_real/`, skip training and load it.

In [ ]:
MODEL_PATH = "models/scvi_model_486k_real"

if os.path.exists(MODEL_PATH):
    print(f"Loading existing scVI model from {MODEL_PATH}...")
    scvi.model.SCVI.setup_anndata(
        adata,
        layer='counts',
        batch_key='dataset_id' if 'dataset_id' in adata.obs.columns else None,
        categorical_covariate_keys=['suspension_type'] if 'suspension_type' in adata.obs.columns else None,
    )
    model = scvi.model.SCVI.load(MODEL_PATH, adata=adata)
else:
    print("Training new scVI model (v26 config)...")
    scvi.model.SCVI.setup_anndata(
        adata,
        layer='counts',
        batch_key='dataset_id' if 'dataset_id' in adata.obs.columns else None,
        categorical_covariate_keys=['suspension_type'] if 'suspension_type' in adata.obs.columns else None,
    )
    model = scvi.model.SCVI(
        adata,
        n_hidden=256,
        n_latent=30,
        n_layers=2,
        gene_likelihood='nb',
        use_layer_norm='both',
    )
    model.train(
        max_epochs=400,
        early_stopping=True,
        early_stopping_patience=30,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    )
    model.save(MODEL_PATH, overwrite=True)
    print(f"Saved to {MODEL_PATH}")

print(f"scVI model ready. Latent dim: {model.n_latent}")

## 5. Extract Neural Markers & Build Training Data

For each cell, decode the scVI latent to get the 4000-gene expression profile,
then extract the neural marker genes. Group by donor age to build the training set.

In [ ]:
# Get the latent representation
latent = model.get_latent_representation()
print(f"Latent shape: {latent.shape}")

# Decode to get denoised expression
print("Decoding expression profiles...")
# For efficiency, sample cells per age group
if age_col:
    # group cells by donor age
    age_groups = adata.obs.groupby(age_col).groups
    n_per_age = min(100, min(len(v) for v in age_groups.values()))
    
    training_data = []
    for age, indices in age_groups.items():
        sample_idx = np.random.choice(indices, n_per_age, replace=False)
        for idx in sample_idx:
            # get decoded expression for this cell
            z = torch.tensor(latent[idx:idx+1])
            with torch.no_grad():
                decoded = model.module.generative(z)['px_rate']
            expr = decoded.squeeze().cpu().numpy()
            # map to gene names
            expr_dict = {gene: float(expr[i]) for i, gene in enumerate(adata.var_names[adata.var['highly_variable']])}
            # extract neural markers
            neural_expr = {g: expr_dict.get(g, 0.0) for g in NEURAL_MARKERS}
            training_data.append({
                'age': float(age),
                'neural_expr': neural_expr,
                'cell_idx': int(idx),
            })
    
    print(f"Training samples: {len(training_data)}")
    print(f"Age range: {min(d['age'] for d in training_data):.0f} - {max(d['age'] for d in training_data):.0f}")
else:
    print("ERROR: No age column — cannot train neural clock")

## 6. Fit the Neural Age Clock

The clock uses a weighted linear model: each neural marker has a decline
rate (from literature) and a weight. We calibrate the weights on the
training data using ridge regression.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict
from scipy.stats import pearsonr

# Build X (marker expression) and y (donor age)
X = np.array([[d['neural_expr'][g] for g in NEURAL_MARKERS] for d in training_data])
y = np.array([d['age'] for d in training_data])

# Log-transform ages (Horvath-style, adult threshold = 40)
F0 = 40.0
y_log = np.where(y <= F0, np.log(y + 1), (y - F0) / 20.0 + np.log(F0 + 1))

# Ridge regression with cross-validation
model_ridge = Ridge(alpha=1.0)
y_pred_log = cross_val_predict(model_ridge, X, y_log, cv=5)

# Transform back to age
y_pred = np.where(y_pred_log <= np.log(F0 + 1),
                   np.exp(y_pred_log) - 1,
                   (y_pred_log - np.log(F0 + 1)) * 20.0 + F0)
y_pred = np.clip(y_pred, 20, 100)

# Evaluate
r, p = pearsonr(y, y_pred)
mae = np.mean(np.abs(y - y_pred))
print(f"Neural Age Clock cross-validation:")
print(f"  Pearson r: {r:.3f} (p={p:.2e})")
print(f"  MAE: {mae:.1f} years")
print(f"  Range: {y_pred.min():.1f} - {y_pred.max():.1f}")

In [ ]:
# Fit final model on all data
model_ridge.fit(X, y_log)

# Extract learned marker weights
marker_weights = {gene: float(model_ridge.coef_[i]) for i, gene in enumerate(NEURAL_MARKERS)}
print("Learned marker weights (regression coefficients):")
for g, w in sorted(marker_weights.items(), key=lambda x: -abs(x[1])):
    print(f"  {g:10s}  {w:+.4f}")

# Plot: predicted vs actual age
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.3, s=10)
ax.plot([20, 100], [20, 100], 'r--', label='y=x')
ax.set_xlabel('Chronological Age')
ax.set_ylabel('Predicted Neural Age')
ax.set_title(f'Neural Age Clock (r={r:.3f}, MAE={mae:.1f}y)')
ax.legend()
plt.tight_layout()
plt.savefig('neural_age_clock_validation.png', dpi=150)
plt.show()

## 7. Save the Trained Clock

Save the weights + configuration for use in `bridge_server.py`.

In [ ]:
import json

clock_config = {
    'version': 'v31_neural_v1',
    'markers': NEURAL_MARKERS,
    'weights': marker_weights,
    'intercept': float(model_ridge.intercept_),
    'age_min': 20,
    'age_max': 100,
    'adult_threshold': 40,
    'training_cells': len(training_data),
    'training_r': float(r),
    'training_mae': float(mae),
    'dataset': 'Human Heart Cell Atlas (Litviňuková 2020)',
    'scvi_model': 'scvi_model_486k_real',
}

SAVE_PATH = 'models/neural_age_clock_v1'
os.makedirs(SAVE_PATH, exist_ok=True)
with open(f'{SAVE_PATH}/config.json', 'w') as f:
    json.dump(clock_config, f, indent=2)

# Also save the sklearn model
import joblib
joblib.dump(model_ridge, f'{SAVE_PATH}/ridge_model.pkl')

print(f"Neural Age Clock saved to {SAVE_PATH}/")
print(json.dumps(clock_config, indent=2))

## 8. Dual-Age Validation (Horvath + Neural)

If you have methylation data for the same donors, validate the dual-age
phenotype classification. Otherwise, simulate for demonstration.

In [ ]:
# Simulate dual-age comparison (replace with real Horvath ages if available)
np.random.seed(42)
horvath_ages = y + np.random.normal(0, 5, len(y))  # simulated Horvath
neural_ages = y_pred  # from our clock

dual_gaps = neural_ages - horvath_ages
phenotypes = []
for gap in dual_gaps:
    if abs(gap) < 3:
        phenotypes.append('concordant')
    elif gap > 3:
        phenotypes.append('neural_dominant')
    else:
        phenotypes.append('genomic_dominant')

from collections import Counter
print("Dual-age phenotype distribution:")
for ph, count in Counter(phenotypes).most_common():
    print(f"  {ph:20s} {count:5d} ({count/len(phenotypes)*100:.0f}%)")

# Plot dual-age scatter
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
colors = {'concordant': 'green', 'neural_dominant': 'red', 'genomic_dominant': 'blue'}
for ph in colors:
    mask = np.array(phenotypes) == ph
    ax.scatter(horvath_ages[mask], neural_ages[mask], alpha=0.4, s=15,
              c=colors[ph], label=f'{ph} ({mask.sum()})')
ax.plot([20, 100], [20, 100], 'k--', alpha=0.3)
ax.set_xlabel('Horvath Age (epigenetic)')
ax.set_ylabel('Neural Age (functional)')
ax.set_title('Dual-Age Phenotype Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('dual_age_phenotypes.png', dpi=150)
plt.show()

## 9. Export for Production

Copy the saved model to your Zenith repo:
```
models/neural_age_clock_v1/
├── config.json       # marker weights + metadata
└── ridge_model.pkl   # sklearn Ridge model
```

Then update `services/neural_age_clock.py` to load these weights instead
of the literature-based defaults.

### Wiring into bridge_server.py:
```python
from routers.neural_router import router as neural_router
app.include_router(neural_router)
```

### The headline metric for marketing:
> **"Zenith v31 — the first platform to measure both epigenetic age and neural age. Dual-age phenotyping reveals whether aging is genomic or autonomic."**